# Inverse Modeling example

We'll show a simple example for inverse modeling using automatic differentiation on a vertex model with periodic boundary conditions (PBC).

**What this notebook does:** it builds a mesh, assigns it random line tensions and equilibrates it (the *forward* problem), builds a second "target" mesh with different tensions, and then tries to recover the target's tensions by nudging the first mesh's parameters so its equilibrium vertex positions match the target's (the *inverse* problem).

**Conventions used here** (see `docs/technical/` for the full justification):
- Mechanical/equilibrium quantities are computed in **float64** (`jax_enable_x64`).
- Line tensions have an intrinsic **additive gauge freedom**: adding the same constant to every tension does not change the mesh's mechanical equilibrium, so it cannot be recovered from geometry alone. We fix it with the **mean-tension gauge** (`mean(Lambda) = 1`, applied automatically after every outer step via `bilevel_optimizer.mean_tension_gauge`), not by pinning one edge's tension to a hardcoded reference value (an older, inferior approach that biases the estimate if the pinned edge is itself noisy).

First, let's check that VertAX is correctly installed by importing all we'll need for the next parts.

In [1]:
from vertax import PbcBilevelOptimizer, PbcMesh
# We'll see how to those important classes.

**Float64.** VertAX's mechanical equilibrium and inverse-problem machinery is validated in float64; JAX defaults to float32. Enable float64 before any mesh or array is created (setting it later can silently leave already-created arrays/traces in float32).

In [2]:
import jax

jax.config.update("jax_enable_x64", True)


Now, let's create our initial mesh. It will have 20 cells, in a square domain.

In [3]:
import math

# Settings
n_cells = 20
# Square domain that can contains 20 cells of area 1.
width = math.sqrt(n_cells)
height = width

# Create a new mesh with periodic boundary conditions, from an initial Voronoi graph with random seeds positions.
mesh = PbcMesh.from_random_seeds(nb_seeds=n_cells, width=width, height=height, random_key=0)



In [4]:
from vertax import plot_mesh

plot_mesh(mesh, title="Initial mesh")

print(f"Created mesh has {mesh.nb_faces} cells, {mesh.nb_edges} edges and {mesh.nb_vertices} vertices.")

Created mesh has 20 cells, 60 edges and 40 vertices.


/Users/Turlier/TURLIERLAB Dropbox/Herve Turlier/Research/projects/vertAX/vertax/src/vertax/meshes/plot.py:282: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The mesh is associated to parameters on vertices, (half-)edges and faces (cells). The meaning of these parameters is "given" by the loss functions.

In our example, we'll only focus on the parameters of the edges, which will be the tensions at given edge ; and the parameters for faces will be their area, fixed to 1.

In [5]:
import jax  # VertAX uses jax and manipulates jax arrays.
import jax.numpy as jnp  # Use it like numpy

# Set mesh parameters
# 1) We don't use vertices, basic initialization.
mesh.vertices_params = jnp.asarray([0.0 for _ in range(mesh.nb_vertices)])

# 2) We give a random tension to each edge.
mu_tensions = 1.2
std_tensions = 0.1

key = jax.random.PRNGKey(1)  # change the seed for different results
he_params = mu_tensions + std_tensions * jax.random.normal(key, shape=(mesh.nb_edges,))
# As we have half-edges, this part is necessary so that corresponding half-edges have the same tension.
mesh.edges_params = jnp.repeat(he_params, 2)

# 3) We set the faces areas to 1.
mesh.faces_params = jnp.asarray([1.0 for _ in range(mesh.nb_faces)])

It is time to define our custom inner loss function. This energy function will penalize cell areas different than 1 and the length of edges depending on the tension on them.

The intended way to get this energy is `vertax.energy.make_energy_line_tensions(K_areas)`, so `K_areas` is explicit and the convention matches every other driver in the repository -- see the note in the next cell for why we inline it by hand here instead.

In [6]:
from jax import Array, vmap

from vertax.geo import get_area, get_length  # geometry functions to compute area/length under PBC
from vertax.opt import InnerLossFunction  # Only here for the type hint, not important

# K_areas sets the relative stiffness of the area term against the line-tension term.
# It is NOT hardcoded inside vertax.energy -- it must be passed explicitly and asserted at every
# call site (a past hardcoded value was a reproducibility hazard). Every periodic (PBC) driver in
# this repository uses K_areas = 10.
K_areas = 10.0
assert K_areas == 10.0  # explicit per the project's standing numerical rule

MAX_EDGES_IN_ANY_FACE = 20  # Assumption: no face has more than 20 edges (performance only).

# NOTE on `vertax.energy.make_energy_line_tensions`: this is the library's *intended* canonical
# factory for exactly this energy (E2 = line tensions + area penalty), and would normally be the
# right thing to call here instead of hand-rolling it. As found while preparing this notebook, it
# currently raises `TypeError: get_length() takes 6 positional arguments but 7 were given`: its
# internal `_hedge_part` helper calls `get_length(..., MAX_EDGES_IN_ANY_FACE)`, but unlike
# `get_area`, `get_length` does not take a max-edges-per-face argument. This looks like a
# copy/paste regression from `_area_part` and was confirmed by direct execution, not just
# inspection -- see the repo-hygiene report. Until it is fixed upstream (src/vertax/energy.py),
# we inline the same energy definition here (identical maths, identical K_areas convention) so
# this notebook actually runs end to end.


def area_part(face: Array, face_param: Array, vertTable: Array, heTable: Array, faceTable: Array) -> Array:
    """Area part of the energy function, per given face."""
    a = get_area(face, vertTable, heTable, faceTable, width, height, MAX_EDGES_IN_ANY_FACE)
    return (a - face_param) ** 2


def hedge_part(he: Array, he_param: Array, vertTable: Array, heTable: Array, faceTable: Array) -> Array:
    """Half-edges take part in the energy function too: tension x length."""
    edge_length = get_length(he, vertTable, heTable, faceTable, width, height)
    return he_param * edge_length


def energy(
    vertTable: Array,
    heTable: Array,
    faceTable: Array,
    _vert_params: Array,
    he_params: Array,
    face_params: Array,
) -> Array:
    """E2 energy: elastic penalty on cell areas + line tensions x edge lengths.

    Every edge is treated identically (no reference/pinned edge), so the same function is reused
    unchanged for both the source mesh and the target mesh below; the tension gauge is fixed
    separately via `mean_tension_gauge`, not by hardcoding one edge's tension here.
    """

    def mapped_areas_part(face: Array, face_param: Array) -> Array:
        return area_part(face, face_param, vertTable, heTable, faceTable)

    def mapped_hedges_part(he: Array, he_param: Array) -> Array:
        return hedge_part(he, he_param, vertTable, heTable, faceTable)

    areas_part = vmap(mapped_areas_part)(jnp.arange(len(faceTable)), face_params)
    hedges_part = vmap(mapped_hedges_part)(jnp.arange(len(heTable)), he_params)
    return (0.5 * jnp.sum(hedges_part)) + (0.5 * K_areas) * jnp.sum(areas_part)


Now we are ready to perform the inner optimization ! This part is done by a PbcBilevelOptimizer, that stores hyper-parameters for the optimizations.

Here are some parameters that are taken into account with the inner optimization :

In [7]:
import optax  # Defines solvers for the optimization problems

bilevel_optimizer = PbcBilevelOptimizer()
bilevel_optimizer.loss_function_inner = energy  # Don't forget to set the loss function !
# Note: those are base values so the following can be omitted
bilevel_optimizer.inner_solver = optax.sgd(learning_rate=0.01)  # inner solver
bilevel_optimizer.update_T1 = True # Perform T1 transitions if necessary.
bilevel_optimizer.min_dist_T1 = 0.005  # Threshold to perform the T1 transitions.
bilevel_optimizer.max_nb_iterations = 1000  # Upper bound for the number of minimization operations
bilevel_optimizer.tolerance = 1e-4  # Below this level, we consider that the loss is stagnating
bilevel_optimizer.patience = 5  # Maximum number of consecutive stagnating loss before we stop.

Let's perform the inner optimization. It's as simple as that:

In [8]:
# Another parameter is beta (for EP).
loss_history = bilevel_optimizer.inner_optimization(mesh)
# If you want to select only a subset of vertices, edges, and faces, it's possible:
# bilevel_optimizer.inner_optimization(mesh,
#     only_on_vertices=[list_vertex_ids],
#     only_on_edges=[list_edges_id],
#     only_on_faces=[list_faces_id],
# )

In [9]:
plot_mesh(mesh, title="The mesh after the inner optimization.")

/Users/Turlier/TURLIERLAB Dropbox/Herve Turlier/Research/projects/vertAX/vertax/src/vertax/meshes/plot.py:282: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The inner optimization nudged the mesh to minimize the length of edges with high tension while keeping the area of each face close to 1.

We'll use a bilevel optimization method to make this mesh correspond to another as much as possible, while keeping the constraints on the tensions and areas.

Let's set this "target" mesh.

The principle is the same :

In [10]:
# We copy all the parameters from previous mesh to create a new mesh.
target = PbcMesh.copy_mesh(mesh)

# We change the tensions (edges parameters) of the target mesh.
key = jax.random.PRNGKey(2)  # change the seed for different target tensions
he_params_target = mu_tensions + std_tensions * jax.random.normal(key, shape=(target.nb_edges,))
target.edges_params = jnp.repeat(he_params_target, 2)

# Energy minimization (target equilibrium). Note: we reuse the same `energy` function defined above
# unchanged -- there is no reference edge to swap out, because the gauge is fixed additively via
# `mean_tension_gauge` further down, not by pinning any one edge's tension.
target_loss_history = bilevel_optimizer.inner_optimization(target)


In [11]:
plot_mesh(target, title="The target mesh is a little bit different.")

OK now we obtained our target vertices, edges and faces, let's store them in our mesh : they become the mesh target.

In [12]:
bilevel_optimizer.vertices_target = target.vertices.copy()
bilevel_optimizer.edges_target = target.edges.copy()
bilevel_optimizer.faces_target = target.faces.copy()
# We can discard the target mesh now because it is of no use.
# del target

# No need to redefine or swap the inner energy function for the target mesh: `energy` treats every
# edge identically (see the cell that defines it), so the same closure already assigned via
# `bilevel_optimizer.loss_function_inner = energy` above is reused for both meshes.


In addition to the previous parameters for the inner optimization, the following parameters are used during the bilevel optimization process:

In [13]:
from vertax.cost import cost_v2v  # We use a loss function defined by vertax, but you can use yours.

# This one compute the distance of each vertices between the mesh and the target.
bilevel_optimizer.loss_function_outer = cost_v2v

from vertax import BilevelOptimizationMethod  # An enumeration of the methods you can use

# Once again this part can be omitted because those are the base value for these parameters
bilevel_optimizer.outer_solver = optax.adam(learning_rate=0.0001, nesterov=True)  # outer solver
bilevel_optimizer.bilevel_optimization_method = BilevelOptimizationMethod.EQUILIBRIUM_PROPAGATION

# Line tensions carry an additive gauge freedom (adding a constant to every tension does not change
# the mechanical equilibrium), so it is not identifiable from geometry and must be fixed by
# convention. We fix mean(Lambda) = 1 by post-step parameter renormalisation, applied after every
# outer step -- see `bilevelopt.py` (`outer_optimization`) for why this is preferred over a fixed
# reference edge or a regularisation penalty (the latter is mis-scaled against a filtered
# hypergradient when the inner solve is truncated).
bilevel_optimizer.mean_tension_gauge = 1.0


Let's run our bilevel optimization process. The first epoch is always a bit long but the next ones will be quick.

In a real simulation, a high number of epochs is necessary for a good result. Here, for this basic tutorial, we'll only use a few epochs.

(Also note that the performances are always worse in a notebook !)

In [14]:
from time import perf_counter

nb_epochs = 10

cost_history = []
for epoch in range(nb_epochs + 1):
    t_start = perf_counter()
    # The cost used compares the vertices positions only.
    # Other parameters are kept because it is important to always use the same signature function.
    cost = cost_v2v(
                mesh.vertices,
                mesh.edges,
                mesh.faces,
                mesh.width,
                mesh.height,
                bilevel_optimizer.vertices_target,
                bilevel_optimizer.edges_target,
                bilevel_optimizer.faces_target,
            )
    cost_history.append(cost)
    print(f"epoch: {epoch}/{nb_epochs}, actual cost is {cost}")

    loss_history = bilevel_optimizer.bilevel_optimization(mesh)
    print(f"epoch {epoch} took {perf_counter() - t_start} seconds.")
    pearson_corr = float(jnp.corrcoef(mesh.edges_params, target.edges_params)[0, 1])
    print(pearson_corr)

epoch: 0/10, actual cost is 0.0030173982959240675


[vertax] WARNING: beta was never set; EQUILIBRIUM_PROPAGATION is using the default beta=0.01. Set it explicitly and record it with your results -- note the manuscript convention is 2x the code value.


epoch 0 took 3.098670959006995 seconds.
0.0397615011180904
epoch: 1/10, actual cost is 0.002952831331640482


epoch 1 took 1.005153083242476 seconds.
0.04051879654906725
epoch: 2/10, actual cost is 0.002932540839537978


epoch 2 took 1.0066811670549214 seconds.
0.04130904402707091
epoch: 3/10, actual cost is 0.0029128023888915777


epoch 3 took 0.9985723746940494 seconds.
0.04204248372256243
epoch: 4/10, actual cost is 0.0028944136574864388


... (3 outputs left out)

epoch 7 took 0.9926572917029262 seconds.
0.04501461931394155
epoch: 8/10, actual cost is 0.0028352593071758747


epoch 8 took 0.9885750408284366 seconds.
0.04569213417781141
epoch: 9/10, actual cost is 0.002824188442900777


epoch 9 took 1.0437707910314202 seconds.
0.0464392603311392
epoch: 10/10, actual cost is 0.00281452271156013


epoch 10 took 0.9924317919649184 seconds.
0.04711765535541282


Just to check that the process is working, let's do a basic plot of the cost in function of the epoch.

In [15]:
import matplotlib.pyplot as plt

plt.plot(range(nb_epochs+1), cost_history)
plt.show()

/var/folders/kb/9_kphn1534b1s0d1kq45s5kw0000gr/T/ipykernel_55681/1560349139.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Of course, a higher number of epochs is necessary for the cost function to stabilize and for a result to be visible in the mesh. If we plot the mesh after a sufficient number of epochs, we'll see that the vertices moved to match the best it can the target while respecting tensions and area constraints.

In [16]:
plot_mesh(mesh, title=f"The mesh after {nb_epochs} epochs")

/Users/Turlier/TURLIERLAB Dropbox/Herve Turlier/Research/projects/vertAX/vertax/src/vertax/meshes/plot.py:282: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


We can also plot the tensions and areas:

In [17]:
from vertax import EdgePlot, FacePlot, VertexPlot  # Enumerations to decide what to show on the plot.

plot_mesh(mesh, edge_plot=EdgePlot.EDGE_PARAMETER, edge_parameters_name="tension",
          face_plot=FacePlot.FACE_PARAMETER, face_parameters_name="area",
          vertex_plot=VertexPlot.INVISIBLE,
          title="Area of cells and tensions on edges.")

In [18]:
pearson_corr = float(jnp.corrcoef(mesh.faces_params, target.faces_params)[0, 1])
print(pearson_corr)
print(target.faces_params)
print(mesh.faces_params)
print(len(mesh.faces_params))
print(len(target.faces_params))
print((jnp.corrcoef(mesh.faces_params, target.faces_params)))
print((jnp.cov(mesh.faces_params, target.faces_params)))

nan
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[0.99985254 0.99867364 1.00162093 1.00103121 1.00132611 1.00162095
 1.00132746 0.99896838 1.00162087 1.00014745 0.99837904 0.99837911
 1.00044192 0.99955821 1.00162057 0.99837909 0.99926316 0.99867383
 0.99926282 1.00162099]
20
20
[[ 1. nan]
 [nan nan]]
[[1.66044439e-06 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00]]


In [19]:
pearson_corr = float(jnp.corrcoef(mesh.edges_params, target.edges_params)[0, 1])
print(pearson_corr)

0.04711765535541282
